# Week 5 — Baseline Model Development
### Datasets: German Credit · LendingClub · Credit Card Fraud Detection

Loads processed parquet splits saved by Week 4.  
Trains Logistic Regression, Random Forest, XGBoost on each dataset.  
Evaluates with Accuracy, Precision, Recall, F1, ROC-AUC, KS Statistic.  
Logs all runs to MLflow. Saves best model registry for Week 6 XAI.

> Run cells top-to-bottom. Week 4 must have been run first.

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

## Cell 2 — Install Dependencies

In [ ]:
!pip install -q xgboost==2.1.1 mlflow==2.16.2 pyarrow==16.1.0
print('Done.')

## Cell 3 — Imports & Paths

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import mlflow
import xgboost as xgb
warnings.filterwarnings('ignore')

from sklearn.linear_model  import LogisticRegression
from sklearn.ensemble      import RandomForestClassifier
from sklearn.metrics       import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', None)
plt.rcParams['figure.dpi'] = 110

# ── Paths (matching Week 4 output folders exactly) ───────────────────
WK4_ROOT   = '/content/drive/MyDrive/CreditRisk_Wk4'
PROCESSED  = f'{WK4_ROOT}/processed'
MODELS_DIR = f'{WK4_ROOT}/models'
os.makedirs(MODELS_DIR, exist_ok=True)

mlflow.set_tracking_uri(f'file://{WK4_ROOT}/mlruns')

# Verify all three split folders exist before proceeding
for ds in ['german', 'lendingclub', 'fraud']:
    path = f'{PROCESSED}/{ds}'
    status = '✅' if os.path.exists(path) else '❌ MISSING — re-run Week 4'
    print(f'  {ds:15s} {status}')

## Cell 4 — Helpers

In [ ]:
def load_splits(dataset_name):
    base = f'{PROCESSED}/{dataset_name}'
    X_train = pd.read_parquet(f'{base}/X_train.parquet')
    X_val   = pd.read_parquet(f'{base}/X_val.parquet')
    X_test  = pd.read_parquet(f'{base}/X_test.parquet')
    y_train = pd.read_parquet(f'{base}/y_train.parquet')['target']
    y_val   = pd.read_parquet(f'{base}/y_val.parquet')['target']
    y_test  = pd.read_parquet(f'{base}/y_test.parquet')['target']
    print(f'[{dataset_name}] train={X_train.shape}  val={X_val.shape}  test={X_test.shape}')
    return X_train, X_val, X_test, y_train, y_val, y_test


def ks_statistic(y_true, y_prob):
    y_true = np.array(y_true)
    pos = np.sort(y_prob[y_true == 1])
    neg = np.sort(y_prob[y_true == 0])
    thresh = np.unique(np.concatenate([pos, neg]))
    tpr = np.array([np.mean(pos <= t) for t in thresh])
    fpr = np.array([np.mean(neg <= t) for t in thresh])
    return float(np.max(np.abs(tpr - fpr)))


def evaluate(model, X, y, split_name='val', threshold=0.5):
    y_prob = model.predict_proba(X)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)
    return {
        f'{split_name}_accuracy' : round(accuracy_score(y, y_pred),  4),
        f'{split_name}_precision': round(precision_score(y, y_pred, zero_division=0), 4),
        f'{split_name}_recall'   : round(recall_score(y, y_pred,    zero_division=0), 4),
        f'{split_name}_f1'       : round(f1_score(y, y_pred,        zero_division=0), 4),
        f'{split_name}_roc_auc'  : round(roc_auc_score(y, y_prob),  4),
        f'{split_name}_ks'       : round(ks_statistic(y.values, y_prob), 4),
    }


def log_model_run(exp_name, run_name, params, val_metrics, test_metrics,
                  model, dataset_name, model_tag):
    mlflow.set_experiment(exp_name)
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics({**val_metrics, **test_metrics})
        mlflow.set_tag('dataset', dataset_name)
        mlflow.set_tag('model',   model_tag)
        model_path = f'{MODELS_DIR}/{dataset_name}_{model_tag}.joblib'
        joblib.dump(model, model_path)
        mlflow.log_artifact(model_path)
        print(f'  MLflow logged: {run_name}')


def plot_evaluation(model, X_val, y_val, X_test, y_test, title=''):
    def get(X):
        probs = model.predict_proba(X)[:, 1]
        return probs, (probs >= 0.5).astype(int)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(title, fontsize=13, weight='bold')

    probs_val, preds_val = get(X_val)
    ConfusionMatrixDisplay(confusion_matrix(y_val, preds_val)).plot(ax=axes[0], colorbar=False)
    axes[0].set_title('Confusion Matrix (Val)')

    probs_te, preds_te = get(X_test)
    ConfusionMatrixDisplay(confusion_matrix(y_test, preds_te)).plot(ax=axes[1], colorbar=False)
    axes[1].set_title('Confusion Matrix (Test)')

    fpr_v, tpr_v, _ = roc_curve(y_val,  probs_val)
    fpr_t, tpr_t, _ = roc_curve(y_test, probs_te)
    axes[2].plot(fpr_v, tpr_v, label=f'Val  (AUC={roc_auc_score(y_val,  probs_val):.3f})')
    axes[2].plot(fpr_t, tpr_t, label=f'Test (AUC={roc_auc_score(y_test, probs_te):.3f})')
    axes[2].plot([0,1],[0,1],'k--', lw=1)
    axes[2].set_xlabel('FPR'); axes[2].set_ylabel('TPR')
    axes[2].set_title('ROC Curve'); axes[2].legend()

    plt.tight_layout(); plt.show()


def compare_models(results_dict, dataset_label):
    """Print highlighted comparison table and return name of best model."""
    df = pd.DataFrame(results_dict).set_index('model')
    print(f'\n═══ {dataset_label} — All Models ═══')
    display(df.style.highlight_max(axis=0, color='#d4edda',
            subset=[c for c in df.columns if 'test' in c]))
    best = df['test_roc_auc'].idxmax()
    print(f'\n✅ Best: {best}  '
          f'AUC={df.loc[best,"test_roc_auc"]:.4f}  '
          f'Acc={df.loc[best,"test_accuracy"]:.4f}  '
          f'F1={df.loc[best,"test_f1"]:.4f}  '
          f'KS={df.loc[best,"test_ks"]:.4f}')
    return best, df


TAG_MAP = {'Logistic Regression': 'logreg', 'Random Forest': 'rf', 'XGBoost': 'xgb'}
print('Helpers ready.')

---
# Section 1 — German Credit

In [ ]:
X_tr1, X_va1, X_te1, y_tr1, y_va1, y_te1 = load_splits('german')

## 1.1 Logistic Regression

In [ ]:
lr1_params = dict(C=0.1, solver='lbfgs', max_iter=1000,
                  class_weight='balanced', random_state=RANDOM_STATE)
lr1 = LogisticRegression(**lr1_params).fit(X_tr1, y_tr1)
lr1_val  = evaluate(lr1, X_va1, y_va1, 'val')
lr1_test = evaluate(lr1, X_te1, y_te1, 'test')
print(pd.DataFrame([lr1_val, lr1_test], index=['val','test']).T)
print(classification_report(y_te1, (lr1.predict_proba(X_te1)[:,1]>=0.5).astype(int)))
log_model_run('wk5_baseline_models', 'german_logreg',
              {**lr1_params, 'dataset':'german'},
              lr1_val, lr1_test, lr1, 'german', 'logreg')
plot_evaluation(lr1, X_va1, y_va1, X_te1, y_te1, 'German Credit — Logistic Regression')

## 1.2 Random Forest

In [ ]:
rf1_params = dict(n_estimators=300, max_depth=8, min_samples_leaf=5,
                  class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
rf1 = RandomForestClassifier(**rf1_params).fit(X_tr1, y_tr1)
rf1_val  = evaluate(rf1, X_va1, y_va1, 'val')
rf1_test = evaluate(rf1, X_te1, y_te1, 'test')
print(pd.DataFrame([rf1_val, rf1_test], index=['val','test']).T)
print(classification_report(y_te1, (rf1.predict_proba(X_te1)[:,1]>=0.5).astype(int)))
log_model_run('wk5_baseline_models', 'german_rf',
              {**rf1_params, 'dataset':'german'},
              rf1_val, rf1_test, rf1, 'german', 'rf')
plot_evaluation(rf1, X_va1, y_va1, X_te1, y_te1, 'German Credit — Random Forest')

## 1.3 XGBoost

In [ ]:
spw1 = float((y_tr1==0).sum() / (y_tr1==1).sum())
xgb1_params = dict(n_estimators=500, max_depth=5, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8,
                   scale_pos_weight=spw1, eval_metric='logloss',
                   random_state=RANDOM_STATE, n_jobs=-1)
xgb1 = xgb.XGBClassifier(**xgb1_params)
xgb1.fit(X_tr1, y_tr1, eval_set=[(X_va1, y_va1)], verbose=False)
xgb1_val  = evaluate(xgb1, X_va1, y_va1, 'val')
xgb1_test = evaluate(xgb1, X_te1, y_te1, 'test')
print(pd.DataFrame([xgb1_val, xgb1_test], index=['val','test']).T)
print(classification_report(y_te1, (xgb1.predict_proba(X_te1)[:,1]>=0.5).astype(int)))
log_model_run('wk5_baseline_models', 'german_xgb',
              {**xgb1_params, 'dataset':'german', 'scale_pos_weight':round(spw1,3)},
              xgb1_val, xgb1_test, xgb1, 'german', 'xgb')
plot_evaluation(xgb1, X_va1, y_va1, X_te1, y_te1, 'German Credit — XGBoost')

## 1.4 Comparison (German Credit)

In [ ]:
best_g, german_results = compare_models([
    {'model':'Logistic Regression', **lr1_val,  **lr1_test},
    {'model':'Random Forest',       **rf1_val,  **rf1_test},
    {'model':'XGBoost',             **xgb1_val, **xgb1_test},
], 'German Credit')

best_model_g = {'Logistic Regression':lr1, 'Random Forest':rf1, 'XGBoost':xgb1}[best_g]
if hasattr(best_model_g, 'feature_importances_'):
    fi = pd.Series(best_model_g.feature_importances_, index=X_tr1.columns)
    fi.nlargest(20).sort_values().plot.barh(
        figsize=(8,5), title=f'{best_g} — Top 20 Features (German Credit)')
    plt.tight_layout(); plt.show()

---
# Section 2 — LendingClub

In [ ]:
X_tr2, X_va2, X_te2, y_tr2, y_va2, y_te2 = load_splits('lendingclub')

## 2.1 Logistic Regression

In [ ]:
lr2_params = dict(C=0.1, solver='saga', max_iter=1000,
                  class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
lr2 = LogisticRegression(**lr2_params).fit(X_tr2, y_tr2)
lr2_val  = evaluate(lr2, X_va2, y_va2, 'val')
lr2_test = evaluate(lr2, X_te2, y_te2, 'test')
print(pd.DataFrame([lr2_val, lr2_test], index=['val','test']).T)
print(classification_report(y_te2, (lr2.predict_proba(X_te2)[:,1]>=0.5).astype(int)))
log_model_run('wk5_baseline_models', 'lendingclub_logreg',
              {**lr2_params, 'dataset':'lendingclub'},
              lr2_val, lr2_test, lr2, 'lendingclub', 'logreg')
plot_evaluation(lr2, X_va2, y_va2, X_te2, y_te2, 'LendingClub — Logistic Regression')

## 2.2 Random Forest

In [ ]:
rf2_params = dict(n_estimators=300, max_depth=10, min_samples_leaf=10,
                  class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
rf2 = RandomForestClassifier(**rf2_params).fit(X_tr2, y_tr2)
rf2_val  = evaluate(rf2, X_va2, y_va2, 'val')
rf2_test = evaluate(rf2, X_te2, y_te2, 'test')
print(pd.DataFrame([rf2_val, rf2_test], index=['val','test']).T)
print(classification_report(y_te2, (rf2.predict_proba(X_te2)[:,1]>=0.5).astype(int)))
log_model_run('wk5_baseline_models', 'lendingclub_rf',
              {**rf2_params, 'dataset':'lendingclub'},
              rf2_val, rf2_test, rf2, 'lendingclub', 'rf')
plot_evaluation(rf2, X_va2, y_va2, X_te2, y_te2, 'LendingClub — Random Forest')

## 2.3 XGBoost

In [ ]:
spw2 = float((y_tr2==0).sum() / (y_tr2==1).sum())
xgb2_params = dict(n_estimators=500, max_depth=6, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8,
                   scale_pos_weight=spw2, eval_metric='logloss',
                   random_state=RANDOM_STATE, n_jobs=-1)
xgb2 = xgb.XGBClassifier(**xgb2_params)
xgb2.fit(X_tr2, y_tr2, eval_set=[(X_va2, y_va2)], verbose=False)
xgb2_val  = evaluate(xgb2, X_va2, y_va2, 'val')
xgb2_test = evaluate(xgb2, X_te2, y_te2, 'test')
print(pd.DataFrame([xgb2_val, xgb2_test], index=['val','test']).T)
print(classification_report(y_te2, (xgb2.predict_proba(X_te2)[:,1]>=0.5).astype(int)))
log_model_run('wk5_baseline_models', 'lendingclub_xgb',
              {**xgb2_params, 'dataset':'lendingclub', 'scale_pos_weight':round(spw2,3)},
              xgb2_val, xgb2_test, xgb2, 'lendingclub', 'xgb')
plot_evaluation(xgb2, X_va2, y_va2, X_te2, y_te2, 'LendingClub — XGBoost')

## 2.4 Comparison (LendingClub)

In [ ]:
best_lc, lending_results = compare_models([
    {'model':'Logistic Regression', **lr2_val,  **lr2_test},
    {'model':'Random Forest',       **rf2_val,  **rf2_test},
    {'model':'XGBoost',             **xgb2_val, **xgb2_test},
], 'LendingClub')

best_model_lc = {'Logistic Regression':lr2, 'Random Forest':rf2, 'XGBoost':xgb2}[best_lc]
if hasattr(best_model_lc, 'feature_importances_'):
    fi = pd.Series(best_model_lc.feature_importances_, index=X_tr2.columns)
    fi.nlargest(20).sort_values().plot.barh(
        figsize=(8,5), title=f'{best_lc} — Top 20 Features (LendingClub)')
    plt.tight_layout(); plt.show()

---
# Section 3 — Credit Card Fraud Detection

In [ ]:
X_tr3, X_va3, X_te3, y_tr3, y_va3, y_te3 = load_splits('fraud')

## 3.1 Logistic Regression

In [ ]:
lr3_params = dict(C=1.0, solver='lbfgs', max_iter=500,
                  class_weight='balanced', random_state=RANDOM_STATE)
lr3 = LogisticRegression(**lr3_params).fit(X_tr3, y_tr3)
lr3_val  = evaluate(lr3, X_va3, y_va3, 'val')
lr3_test = evaluate(lr3, X_te3, y_te3, 'test')
print(pd.DataFrame([lr3_val, lr3_test], index=['val','test']).T)
print(classification_report(y_te3, (lr3.predict_proba(X_te3)[:,1]>=0.5).astype(int)))
log_model_run('wk5_baseline_models', 'fraud_logreg',
              {**lr3_params, 'dataset':'fraud'},
              lr3_val, lr3_test, lr3, 'fraud', 'logreg')
plot_evaluation(lr3, X_va3, y_va3, X_te3, y_te3, 'Credit Card Fraud — Logistic Regression')

## 3.2 Random Forest

In [ ]:
rf3_params = dict(n_estimators=200, max_depth=12, min_samples_leaf=5,
                  class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
rf3 = RandomForestClassifier(**rf3_params).fit(X_tr3, y_tr3)
rf3_val  = evaluate(rf3, X_va3, y_va3, 'val')
rf3_test = evaluate(rf3, X_te3, y_te3, 'test')
print(pd.DataFrame([rf3_val, rf3_test], index=['val','test']).T)
print(classification_report(y_te3, (rf3.predict_proba(X_te3)[:,1]>=0.5).astype(int)))
log_model_run('wk5_baseline_models', 'fraud_rf',
              {**rf3_params, 'dataset':'fraud'},
              rf3_val, rf3_test, rf3, 'fraud', 'rf')
plot_evaluation(rf3, X_va3, y_va3, X_te3, y_te3, 'Credit Card Fraud — Random Forest')

## 3.3 XGBoost

In [ ]:
spw3 = float((y_tr3==0).sum() / (y_tr3==1).sum())
xgb3_params = dict(n_estimators=500, max_depth=6, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8,
                   scale_pos_weight=spw3, eval_metric='logloss',
                   random_state=RANDOM_STATE, n_jobs=-1)
xgb3 = xgb.XGBClassifier(**xgb3_params)
xgb3.fit(X_tr3, y_tr3, eval_set=[(X_va3, y_va3)], verbose=False)
xgb3_val  = evaluate(xgb3, X_va3, y_va3, 'val')
xgb3_test = evaluate(xgb3, X_te3, y_te3, 'test')
print(pd.DataFrame([xgb3_val, xgb3_test], index=['val','test']).T)
print(classification_report(y_te3, (xgb3.predict_proba(X_te3)[:,1]>=0.5).astype(int)))
log_model_run('wk5_baseline_models', 'fraud_xgb',
              {**xgb3_params, 'dataset':'fraud', 'scale_pos_weight':round(spw3,3)},
              xgb3_val, xgb3_test, xgb3, 'fraud', 'xgb')
plot_evaluation(xgb3, X_va3, y_va3, X_te3, y_te3, 'Credit Card Fraud — XGBoost')

## 3.4 Comparison (Fraud)

In [ ]:
best_f, fraud_results = compare_models([
    {'model':'Logistic Regression', **lr3_val,  **lr3_test},
    {'model':'Random Forest',       **rf3_val,  **rf3_test},
    {'model':'XGBoost',             **xgb3_val, **xgb3_test},
], 'Credit Card Fraud')

best_model_f = {'Logistic Regression':lr3, 'Random Forest':rf3, 'XGBoost':xgb3}[best_f]
if hasattr(best_model_f, 'feature_importances_'):
    fi = pd.Series(best_model_f.feature_importances_, index=X_tr3.columns)
    fi.nlargest(20).sort_values().plot.barh(
        figsize=(8,5), title=f'{best_f} — Top 20 Features (Fraud)')
    plt.tight_layout(); plt.show()

---
# Section 4 — Cross-Dataset Summary & Registry

In [ ]:
all_rows = []
for label, res in [('German Credit', german_results),
                   ('LendingClub',   lending_results),
                   ('Fraud',         fraud_results)]:
    for m in res.index:
        all_rows.append({'Dataset': label, 'Model': m, **res.loc[m]})

summary = pd.DataFrame(all_rows).set_index(['Dataset', 'Model'])
print('\n══════════ WEEK 5 — ALL RESULTS ══════════')
display(summary.style.highlight_max(axis=0, color='#d4edda',
        subset=[c for c in summary.columns if 'test' in c]))

print('\n── Accuracy ≥ 85% check ──')
print(summary['test_accuracy'].apply(
    lambda v: f"{v:.1%}  {'✅' if v >= 0.85 else '⚠️  below target'}"
))

In [ ]:
BEST_MODELS = {}
for ds_key, res in [('german',      german_results),
                    ('lendingclub',  lending_results),
                    ('fraud',        fraud_results)]:
    best = res['test_roc_auc'].idxmax()
    BEST_MODELS[ds_key] = {
        'model_name'    : best,
        'model_path'    : f"{MODELS_DIR}/{ds_key}_{TAG_MAP[best]}.joblib",
        'test_roc_auc'  : float(res.loc[best, 'test_roc_auc']),
        'test_accuracy' : float(res.loc[best, 'test_accuracy']),
        'test_f1'       : float(res.loc[best, 'test_f1']),
        'test_ks'       : float(res.loc[best, 'test_ks']),
    }

registry_path = f'{WK4_ROOT}/best_models_registry.json'
with open(registry_path, 'w') as f:
    json.dump(BEST_MODELS, f, indent=2)

print(f'Registry saved → {registry_path}')
print(json.dumps(BEST_MODELS, indent=2))

---
# Summary

| Dataset | Models Trained | Best (by AUC) | Saved |
|---|---|---|---|
| German Credit | LR · RF · XGBoost | auto-selected | `models/german_<tag>.joblib` |
| LendingClub | LR · RF · XGBoost | auto-selected | `models/lendingclub_<tag>.joblib` |
| Credit Card Fraud | LR · RF · XGBoost | auto-selected | `models/fraud_<tag>.joblib` |

**Saved under `CreditRisk_Wk4/`:**
- `models/` — all 9 trained models
- `best_models_registry.json` — loaded by Week 6 for XAI

**MLflow experiment:** `wk5_baseline_models`  
**Next:** Week 6 — SHAP + LIME explanations on the best models.